# RAG 파이프라인 — 규정·지침 HWP/HWPX 문서 QA 시스템 [Lab]
> **핵심 목표**: `data/Regulations_Guidelines` 의 HWP/HWPX 규정 문서를 인덱싱하고, 관련 조항을 검색해 Claude가 근거 기반으로 답변하는 RAG 시스템 구현

**파이프라인 구조**

```
문서 로더(HWP/HWPX) → 텍스트 스플리터 → 임베딩 → 벡터 저장소(Milvus)
        → RAG 체인(LCEL, Claude) → 소스 문서 반환 → 스트리밍 답변
```

| 단계 | 사용 컴포넌트 |
|------|---------------|
| 문서 로드 | `langchain-hwp-hwpx-loader` (`HwpHwpxDirectoryLoader`) |
| 문서 분할 | `RecursiveCharacterTextSplitter` |
| 임베딩 | `HuggingFaceEmbeddings` (로컬 다국어 모델, 384차원) |
| 벡터 저장 | `langchain_milvus.Milvus` (Docker Milvus Standalone) |
| 생성 | `ChatAnthropic` (Claude) + LCEL |

> 💡 Anthropic(Claude)은 임베딩 API가 없어 로컬 sentence-transformers 모델을 사용합니다.

---

## 0. 환경 설정

필요 패키지 설치 (최초 1회, 터미널에서 실행):

```bash
uv add langchain-anthropic langchain-huggingface sentence-transformers langchain-milvus langchain-text-splitters python-dotenv
uv add langchain-hwp-hwpx-loader
```

- HWP/HWPX 로더: [jaypakdevkr/HWP-Loader](https://github.com/jaypakdevkr/HWP-Loader) (`langchain-hwp-hwpx-loader`, 내부 파서 `hwp-hwpx-parser`는 순수 Python)
- `.env` 에 `ANTHROPIC_API_KEY=sk-ant-...` 가 있어야 합니다.
- Milvus 서버가 꺼져 있으면 먼저 `docker compose -f ./docker-compose-milvus.yml up -d` 로 실행합니다.

In [1]:
# 환경변수 로드 및 경로 설정
from dotenv import load_dotenv
from pathlib import Path
from collections import Counter
import os

load_dotenv()
assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY가 설정되지 않았습니다. .env 파일을 확인하세요."

DATA_DIR = Path("data") / "Regulations_Guidelines"   # 이 노트북 기준 상대 경로
assert DATA_DIR.exists(), f"💡 문서 폴더가 없습니다: {DATA_DIR.resolve()}"

files = sorted(p for p in DATA_DIR.rglob("*") if p.suffix.lower() in (".hwp", ".hwpx"))

def detect_format(path: Path) -> str:
    """확장자가 아니라 파일 첫 바이트(매직 넘버)로 실제 형식을 판별합니다."""
    with open(path, "rb") as f:
        head = f.read(8)
    if head[:4] == b"\xd0\xcf\x11\xe0":
        return "hwp5"      # 바이너리 HWP 5.x (OLE 컨테이너)
    if head[:2] == b"PK":
        return "hwpx"      # HWPX (ZIP + XML)
    if head[:5] == b"<?xml":
        return "hwpml"     # HWPML 2.x (순수 XML, 법제처 다운로드본에서 흔함)
    return "unknown"

print("환경 설정 완료")
print(f"문서 폴더: {DATA_DIR.resolve()}")
print(f"전체 파일 수: {len(files)}개")
print("확장자별:", dict(Counter(p.suffix.lower() for p in files)))
print("실제 형식별:", dict(Counter(detect_format(p) for p in files)))
print("→ 확장자와 실제 형식이 다른 파일이 있어, 로더가 확장자만 보면 읽기에 실패합니다.")

환경 설정 완료
문서 폴더: C:\Users\skstj\04_LLM\data\Regulations_Guidelines
전체 파일 수: 134개
확장자별: {'.hwpx': 5, '.hwp': 129}
실제 형식별: {'hwpx': 49, 'hwp5': 65, 'hwpml': 20}
→ 확장자와 실제 형식이 다른 파일이 있어, 로더가 확장자만 보면 읽기에 실패합니다.


In [2]:
# 필수 패키지 임포트 확인
try:
    import langchain, langchain_anthropic, langchain_huggingface, langchain_milvus, langchain_text_splitters
    import hwp_hwpx_parser
    print("필수 패키지 임포트 성공")
except ImportError as e:
    print(f"패키지 임포트 실패: {e}")
    print("uv add langchain-anthropic langchain-huggingface sentence-transformers langchain-milvus langchain-text-splitters langchain-hwp-hwpx-loader 를 실행하세요.")

필수 패키지 임포트 성공


---

## 1. 문서 로드 (Document Loader)

HWP/HWPX 파서는 [jaypakdevkr/HWP-Loader](https://github.com/jaypakdevkr/HWP-Loader) 가 사용하는 `hwp-hwpx-parser` 를 씁니다.

> ⚠️ **이 폴더의 파일은 확장자와 실제 형식이 다릅니다.** `HwpHwpxDirectoryLoader` 는 확장자만 보고 파서를 고르기 때문에 대부분 `Document skipped because it is invalid` 로 실패했습니다.
> 그래서 **파일 첫 바이트로 실제 형식을 판별**해 알맞은 파서로 읽는 로더를 직접 만듭니다.

| 실제 형식 | 판별 | 사용 파서 |
|-----------|------|-----------|
| HWP 5.x | 첫 바이트 `D0 CF 11 E0` (OLE) | `hwp_hwpx_parser.extract_hwp5` |
| HWPX | 첫 바이트 `PK` (ZIP) | `hwp_hwpx_parser.extract_hwpx` |
| HWPML | 첫 바이트 `<?xml` | 표준 라이브러리 `xml.etree` 로 직접 파싱 (머리말·꼬리말 제외) |

In [3]:
# 실제 형식별 로더 정의 → 폴더 전체 문서 로드
import shutil, tempfile
import xml.etree.ElementTree as ET
from langchain_core.documents import Document
from hwp_hwpx_parser import extract_hwp5, extract_hwpx

_SKIP_TAGS = {"HEADER", "FOOTER", "SHAPEOBJECT", "PICTURE", "EQUATION", "ENDNOTE", "FOOTNOTE"}

def _hwpml_paragraphs(node, out):
    """HWPML 의 문단(P)을 순서대로 모읍니다. 표 셀 안의 문단도 재귀로 포함하고 머리말/꼬리말은 건너뜁니다."""
    for ch in node:
        if ch.tag in _SKIP_TAGS:
            continue
        if ch.tag != "P":
            _hwpml_paragraphs(ch, out)
            continue
        buf = []
        for text_node in ch.findall("TEXT"):
            for x in text_node:
                if x.tag == "CHAR":
                    buf.append("".join(x.itertext()))
                elif x.tag not in _SKIP_TAGS:        # 표 등 컨트롤 안에 중첩된 문단
                    nested = []
                    _hwpml_paragraphs(x, nested)
                    if nested:
                        if buf:
                            out.append("".join(buf)); buf = []
                        out.extend(nested)
        line = "".join(buf).replace("\xa0", " ").strip()
        if line:
            out.append(line)

def extract_hwpml(path: Path) -> str:
    root = ET.fromstring(path.read_bytes())
    out = []
    _hwpml_paragraphs(root.find("BODY"), out)
    return "\n".join(out)

def extract_by_format(path: Path, fmt: str) -> str:
    if fmt == "hwpml":
        return extract_hwpml(path)
    # 파서가 확장자로 형식을 판단하므로, 실제 형식에 맞는 확장자의 임시 사본을 만들어 전달합니다.
    with tempfile.TemporaryDirectory() as tmp:
        tmp_path = Path(tmp) / ("doc.hwp" if fmt == "hwp5" else "doc.hwpx")
        shutil.copyfile(path, tmp_path)
        text, _ = (extract_hwp5 if fmt == "hwp5" else extract_hwpx)(tmp_path)
    return text

def load_regulation_docs(paths):
    """형식을 판별해 Document 목록과 실패 목록(파일명, 사유)을 반환합니다."""
    docs, failed = [], []
    for p in paths:
        fmt = detect_format(p)
        try:
            if fmt == "unknown":
                raise ValueError("지원하지 않는 파일 형식")
            text = extract_by_format(p, fmt)
            docs.append(Document(page_content=text, metadata={
                "source": str(p), "file_name": p.name, "file_type": fmt,
            }))
        except Exception as e:
            failed.append((p.name, f"{type(e).__name__}: {e}"))
    return docs, failed

raw_docs, failed = load_regulation_docs(files)
print(f"로드된 문서 수: {len(raw_docs)} / 전체 파일 수: {len(files)}")
print("형식별:", dict(Counter(d.metadata["file_type"] for d in raw_docs)))

로드된 문서 수: 134 / 전체 파일 수: 134
형식별: {'hwpx': 49, 'hwp5': 65, 'hwpml': 20}


In [4]:
# 로드에 실패한 파일 확인
print(f"로드 실패 파일: {len(failed)}개")
for name, reason in failed[:10]:
    print(f"  - {name[:60]} → {reason[:80]}")

# Document 구조 확인 (가장 긴 문서)
doc = max(raw_docs, key=lambda d: len(d.page_content))
print("\n=== Document 구조 ===")
print("metadata:", doc.metadata)
print("page_content (처음 300자):")
print(doc.page_content[:300])

로드 실패 파일: 0개

=== Document 구조 ===
metadata: {'source': 'data\\Regulations_Guidelines\\(전문) 산업기술혁신사업 공통 운영요령(고시 제2024-218호).hwpx', 'file_name': '(전문) 산업기술혁신사업 공통 운영요령(고시 제2024-218호).hwpx', 'file_type': 'hwp5'}
page_content (처음 300자):
신사업 공통 운영요령』
산업통상자원부 고시 제2024-218호(2024. 12. 30.)
「산업기술혁신촉진법」 및 관련법령에 따른 산업기술혁신사업을 효율적으로 추진하기 위하여 「산업기술혁신사업 공통 운영요령」을 다음과 같이 개정하여 고시한다.
                                      2024. 12. 30.
산업통상자원부 장관
제정 2008.12.29 지식경제부고시 제2008-240호
개정 2009. 8.24 지식경제부고시 제2009-193호
개정 2010. 4. 1. 지식경제부 고시 제2010-71호
개


In [5]:
# 공백 정리 + 내용이 거의 없는 문서(빈 서식 등) 제외
import re

def normalize(text: str) -> str:
    # 연속된 공백은 한 개로, 연속된 줄바꿈은 한 번으로 줄임
    return re.sub(r"\n+", "\n", re.sub(r"[^\S\n]+", " ", text)).strip()

MIN_DOC_CHARS = 200
docs = []
for d in raw_docs:
    d.page_content = normalize(d.page_content)
    if len(d.page_content) >= MIN_DOC_CHARS:
        docs.append(d)

print(f"사용할 문서 수: {len(docs)} (내용 {MIN_DOC_CHARS}자 미만 {len(raw_docs) - len(docs)}개 제외)")
print(f"총 {sum(len(d.page_content) for d in docs):,}자")
for d in docs[:5]:
    print(f"  - {d.metadata['file_name'][:60]}  ({len(d.page_content):,}자)")

사용할 문서 수: 129 (내용 200자 미만 5개 제외)
총 1,155,883자
  - (전문) 산업기술개발장비 통합관리요령(산업부공고 제2024-093호).hwpx  (77,971자)
  - (전문) 산업기술혁신사업 공통 운영요령(고시 제2024-218호).hwpx  (156,687자)
  - (전문) 산업기술혁신사업 기술개발 평가관리지침(예규 제139호).hwpx  (56,267자)
  - (전문) 산업기술혁신사업 보안관리요령.hwp  (18,754자)
  - (전문) 산업기술혁신사업 연구윤리 진실성 확보 등에 관한 요령.hwp  (10,884자)


---

## 2. 텍스트 분할 (Text Splitter) — 조문 인식 `LegalArticleSplitter`

문서 대부분이 **제1장 → 제1조(제목) → ①항 → 1.호 → 가.목** 구조의 조문입니다. 글자 수 기준 `RecursiveCharacterTextSplitter` 는 조문 한가운데서 잘라 "어느 조의 내용인지"를 잃기 쉬워서, 조문 구조를 이용하는 스플리터를 직접 만들어 씁니다.

| 개선점 | 설명 |
|--------|------|
| **조(條) 단위 분할** | `제N조(제목)` 줄에서 새 청크를 시작. 본문 중간의 "제5조에 따라" 같은 인용은 제목 괄호가 없어 경계로 오인하지 않음 |
| **문맥 머리말** | 모든 청크 앞에 `[문서명 > 제1장 총칙 > 제2조(정의)]` 를 붙여, 임베딩·검색·LLM 답변이 출처 위치를 알 수 있게 함 |
| **긴 조문은 항·호·목 경계에서 분할** | `max_chars` 초과 시 ①, 1., 가. 에서 끊고 `(1/3)` 표시와 같은 머리말을 반복 |
| **짧은 조문 병합** | 같은 장의 150자 미만 조문은 이어지는 조문과 합쳐 검색 단위가 너무 잘게 쪼개지지 않게 함 |
| **삭제 조문 제외** | `제5조 삭제` 같은 빈 조문은 인덱스에서 제외 |
| **폴백** | 조문 구조가 없는 문서(별지 서식, 별표)는 `RecursiveCharacterTextSplitter` 사용 |
| **메타데이터** | `chapter`, `article` 을 저장 → 이후 `article == "제114조"` 같은 필터 검색 가능 |

In [6]:
import re
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 제1장 총칙 / 제2절 … / 제3편 … / 부칙 / [별표 1] …  (장·절 제목 줄)
_CHAPTER_RE = re.compile(r"^(제\s?\d+\s?[장절편관](?:의\s?\d+)?\s+\S.*|부\s?칙.*|\[?별표\s?\d*\]?.*|\[?별지\s?\d*\]?.*)$")
# 제12조(정의) / 제12조의2(…) / 제12조 삭제  — 본문 중간 인용("제5조에 따라…")과 구분하려고 제목 괄호나 '삭제'를 요구
_ARTICLE_RE = re.compile(r"^(제\s?\d+\s?조(?:의\s?\d+)?)\s*(\([^)]{1,60}\)|삭제)")
# 항(①), 호(1.), 목(가.) 시작 줄 — 긴 조문을 나눌 때 우선 경계로 사용
_SUB_RE = re.compile(r"^([①-⑳]|\d{1,2}\.\s|[가-하]\.\s)")


class LegalArticleSplitter:
    """조문(제N조) 단위로 분할하는 한국 법령·규정용 텍스트 스플리터.

    - 조(條) 1개 = 청크 1개가 기본이며, 장·절 제목은 이후 조문 청크에 문맥으로 붙습니다.
    - max_chars 를 넘는 긴 조문은 항(①)·호(1.)·목(가.) 경계에서 나누고, 나뉜 모든 청크 앞에 같은 머리말을 반복합니다.
    - 머리말 형식: [문서명 > 제1장 총칙 > 제2조(정의)] → 임베딩과 LLM 답변 모두에서 출처 문맥으로 쓰입니다.
    - 조문 구조가 없는 문서는 RecursiveCharacterTextSplitter 로 폴백합니다.
    """

    def __init__(self, max_chars: int = 1000, min_articles_for_legal: int = 3, fallback_chunk_size: int = 800):
        self.max_chars = max_chars
        self.min_articles = min_articles_for_legal
        self.fallback = RecursiveCharacterTextSplitter(
            chunk_size=fallback_chunk_size, chunk_overlap=150, separators=["\n\n", "\n", " ", ""])

    @staticmethod
    def _doc_title(meta: dict) -> str:
        # 파일명에서 확장자와 뒤쪽 (고시번호)(시행일) 꼬리표를 제거
        stem = Path(meta.get("file_name", "")).stem
        return re.sub(r"\s*\((?:[^()]*고시|[^()]*령|[^()]*부)\).*$|\s*\(제?\d+.*$", "", stem).strip() or stem

    def _units(self, text: str):
        """줄 단위로 훑으며 (장제목, 조 머리, 본문 줄 리스트) 단위를 만듭니다."""
        chapter, units, cur = "", [], None
        for line in (l.strip() for l in text.split("\n")):
            if not line:
                continue
            m = _ARTICLE_RE.match(line)
            if m:
                head = re.match(r"^제\s?\d+\s?조(?:의\s?\d+)?\s*(?:\([^)]{1,60}\)|삭제)", line).group(0)
                head = re.sub(r"^(제\s?\d+\s?조(?:의\s?\d+)?)\s*", lambda x: x.group(1).replace(" ", ""), head)   # '제 5 조' → '제5조', 제목 속 공백은 유지
                cur = {"chapter": chapter, "article": m.group(1).replace(" ", ""), "head": head, "lines": [line]}
                units.append(cur)
                continue
            if _CHAPTER_RE.match(line) and len(line) < 80:
                chapter = line
                cur = {"chapter": chapter, "article": "", "head": "", "lines": [line]}
                units.append(cur)
                continue
            if cur is None:                      # 첫 조문 이전(제목·제정이유·개정이력 등)
                cur = {"chapter": "", "article": "", "head": "머리말", "lines": []}
                units.append(cur)
            cur["lines"].append(line)
        return units

    def _merge_short(self, units, min_body: int = 150):
        """같은 장 안에서 내용이 너무 짧은 조문은 다음 조문과 합쳐 검색 단위가 너무 잘게 쪼개지지 않게 합니다."""
        merged = []
        for u in units:
            prev = merged[-1] if merged else None
            if (prev and u["article"] and prev["article"] and prev["chapter"] == u["chapter"]
                    and sum(len(l) for l in prev["lines"]) < min_body
                    and sum(len(l) for l in prev["lines"] + u["lines"]) < self.max_chars * 0.6):
                prev["lines"] = prev["lines"] + u["lines"]
                prev["head"] = prev["head"].split("~")[0] + "~" + u["head"]
                continue
            merged.append(dict(u))
        return merged

    def _pack(self, lines, budget):
        """줄들을 budget 이내로 묶되, 항·호·목 시작 줄에서 우선 끊습니다."""
        chunks, buf, size = [], [], 0
        for ln in lines:
            if len(ln) > budget:                 # 한 줄이 너무 긴 경우만 재귀 분할
                if buf:
                    chunks.append("\n".join(buf)); buf, size = [], 0
                chunks.extend(self.fallback.split_text(ln))
                continue
            if buf and size + len(ln) + 1 > budget and (_SUB_RE.match(ln) or size > budget * 0.8):
                chunks.append("\n".join(buf)); buf, size = [], 0
            elif buf and size + len(ln) + 1 > budget * 1.3:     # 경계가 없어도 상한 초과 방지
                chunks.append("\n".join(buf)); buf, size = [], 0
            buf.append(ln); size += len(ln) + 1
        if buf:
            chunks.append("\n".join(buf))
        return chunks

    def split_document(self, doc: Document):
        text = doc.page_content
        n_articles = sum(1 for l in text.split("\n") if _ARTICLE_RE.match(l.strip()))
        base = {"source": doc.metadata.get("source", ""), "file_name": doc.metadata.get("file_name", "")}
        title = self._doc_title(doc.metadata)

        if n_articles < self.min_articles:       # 조문형이 아닌 문서(서식·별표 등)는 폴백
            return [Document(page_content=f"[{title}]\n{t}", metadata={**base, "chapter": "", "article": "", "split": "fallback"})
                    for t in self.fallback.split_text(text)]

        out = []
        for u in self._merge_short(self._units(text)):
            body_lines = u["lines"]
            if u["head"] != "머리말" and len(body_lines) == 1 and re.search(r"삭제\s*[<＜(\[]?.*$", body_lines[0]) and len(body_lines[0]) < 40:
                continue                         # '제5조 삭제' 같은 빈 조문은 검색에 도움이 안 되어 제외
            path = " > ".join(p for p in (title, u["chapter"], u["head"]) if p)
            header = f"[{path}]"
            parts = self._pack(body_lines, max(200, self.max_chars - len(header) - 1))
            for k, part in enumerate(parts):
                label = f" ({k + 1}/{len(parts)})" if len(parts) > 1 else ""
                out.append(Document(
                    page_content=f"{header}{label}\n{part}",
                    metadata={**base, "chapter": u["chapter"], "article": u["article"], "split": "article"},
                ))
        return out

    def split_documents(self, docs):
        out = []
        for d in docs:
            out.extend(self.split_document(d))
        return out


text_splitter = LegalArticleSplitter(max_chars=1000)
split_docs = text_splitter.split_documents(docs)

# 제목만 남은 짧은 청크·중복 청크 제거 (검색 상위 자리를 낭비하지 않도록)
MIN_CHUNK_CHARS = 50
seen, chunks = set(), []
for c in split_docs:
    t = c.page_content
    if len(t) < MIN_CHUNK_CHARS or t in seen:
        continue
    seen.add(t)
    chunks.append(c)

sizes = [len(c.page_content) for c in chunks]
n_art = sum(c.metadata["split"] == "article" for c in chunks)
print(f"분할 전 문서 수: {len(docs)} → 분할 후 청크 수: {len(split_docs)} → 정리 후: {len(chunks)}")
print(f"조문 단위 청크 {n_art}개 / 폴백(서식·별표) 청크 {len(chunks) - n_art}개")
print(f"청크 크기: 최소 {min(sizes)} / 평균 {sum(sizes)/len(sizes):.0f} / 최대 {max(sizes)}자")

# 샘플: 조문 청크 하나 확인
sample = next(c for c in chunks if c.metadata["split"] == "article" and c.metadata["article"] == "제2조")
print(f"\n--- 샘플 청크 ---\n{sample.page_content[:400]}\n\n메타데이터: {sample.metadata}")

분할 전 문서 수: 129 → 분할 후 청크 수: 2800 → 정리 후: 2727
조문 단위 청크 2116개 / 폴백(서식·별표) 청크 611개
청크 크기: 최소 50 / 평균 501 / 최대 1217자

--- 샘플 청크 ---
[(전문) 산업기술개발장비 통합관리요령(산업부공고 제2024-093호) > 제1장 총칙 > 제2조(정의)] (1/3)
제2조(정의) ①이 요령에서 사용하는 용어의 뜻은 다음 각 호와 같다.
1. "산업기술개발장비"란(이하 "장비"라 한다) 공통운영요령 제3조의 사업을 통해 도입되는 비소모적 자산으로서 공동활용서비스장비, 단독활용장비, 공동활용허용장비를 총칭한다.
2. "공동활용서비스장비"란 산촉법 제19조의 목적을 달성하기 위하여 구축한 장비로서, 연구개발기관의 장은 대내외에 개방하여 공동으로 사용하여야하며 외부 이용자는 보유기관의 이용승낙을 받아 사용할 수 있다.
3. "단독활용장비"란 산촉법 제11조의 목적을 달성하기 위하여 구축한 장비로서, 연구개발기관의 장이 해당 기술개발 등에 단독으로 활용하기 위하

메타데이터: {'source': 'data\\Regulations_Guidelines\\(전문) 산업기술개발장비 통합관리요령(산업부공고 제2024-093호).hwpx', 'file_name': '(전문) 산업기술개발장비 통합관리요령(산업부공고 제2024-093호).hwpx', 'chapter': '제1장 총칙', 'article': '제2조', 'split': 'article'}


---

## 3. 임베딩 (Embeddings)

로컬 다국어 모델 `paraphrase-multilingual-MiniLM-L12-v2` (384차원)를 사용합니다. 최초 실행 시 모델을 내려받습니다(약 470MB).

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    encode_kwargs={"normalize_embeddings": True},   # COSINE 검색용 단위 벡터
)

sample = "연구개발비 사용 기준에서 간접비는 어떻게 계상하나요?"
vec = embedding_model.embed_query(sample)
print(f"임베딩 벡터 차원: {len(vec)}")
print(f"L2 Norm: {np.linalg.norm(vec):.4f}")

# 의미 유사도 확인
pairs = ["연구비 간접비 산정 기준", "연구개발비에서 간접비를 계산하는 방법", "오늘 점심 메뉴 추천"]
v = embedding_model.embed_documents(pairs)
print(f"\n유사(관련):  {float(np.dot(v[0], v[1])):.4f}")
print(f"유사(무관):  {float(np.dot(v[0], v[2])):.4f}")

c:\Users\skstj\04_LLM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2817.44it/s]


임베딩 벡터 차원: 384
L2 Norm: 1.0000

유사(관련):  0.7256
유사(무관):  0.1951


---

## 4. 벡터 저장소 (Milvus)

In [12]:
from langchain_milvus import Milvus
import time

MILVUS_URI = "http://127.0.0.1:19530"   # Docker Milvus Standalone
COLLECTION = "reg_guidelines"

t0 = time.time()
try:
    vector_store = Milvus.from_documents(
        documents=chunks,
        embedding=embedding_model,
        collection_name=COLLECTION,
        connection_args={"uri": MILVUS_URI},
        index_params={"index_type": "AUTOINDEX", "metric_type": "COSINE"},
        consistency_level="Strong",
        drop_old=True,          # 재실행 시 기존 컬렉션을 지우고 새로 생성
    )
except Exception as e:
    raise RuntimeError(
        "💡 Milvus 연결/저장에 실패했습니다. 'docker compose -f ./docker-compose-milvus.yml up -d' 로 서버를 켠 뒤 다시 시도하세요."
    ) from e

stored = vector_store.client.query(collection_name=COLLECTION, filter="", output_fields=["count(*)"])
print(f"벡터 저장소 구축 완료: {stored[0]['count(*)']}개 청크 ({time.time() - t0:.0f}초)")

벡터 저장소 구축 완료: 2727개 청크 (412초)


In [13]:
# 유사도 검색 테스트
query = "연구개발비 간접비는 어떻게 계상하나요?"
results = vector_store.similarity_search_with_score(query=query, k=5)

print(f"검색 질문: {query}\n")
for i, (doc, score) in enumerate(results, 1):
    print(f"--- 결과 {i} (코사인 유사도: {score:.4f}) | {doc.metadata['file_name']} ---")
    print(doc.page_content[:200], "...\n")

# Retriever로 변환 (RAG 체인의 표준 인터페이스)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})

검색 질문: 연구개발비 간접비는 어떻게 계상하나요?

--- 결과 1 (코사인 유사도: 0.6449) | [별지 제4호서식] 단계보고서(국가연구개발혁신법 시행규칙).hwp ---
[[별지 제4호서식] 단계보고서(국가연구개발혁신법 시행규칙)]
9. 총 연구개발비(해당단계, 해당연도): 연구개발계획서 표지에 기재한 연구개발과제의 총(해당단계, 해당연도) 연구개발비를 기재합니다.
10. 연구개발단계: 해당되는 연구개발과제의 연구개발단계 유형에 [√] 표시합니다.
1) 기초연구단계란 특수한 응용 또는 사업을 직접적 목표로 하지 아니하고 현 ...

--- 결과 2 (코사인 유사도: 0.6419) | [별지 제4호서식] 단계보고서(국가연구개발혁신법 시행규칙).hwp ---
[[별지 제4호서식] 단계보고서(국가연구개발혁신법 시행규칙)]
연구개발계획서 표지에 기재한 연구개발과제의 연구개발비 단계 혹은 연도별 사용계획을 중심으로 간결하게 기재합니다. 6) 사업화 추진 계획: 연구개발과제 성과의 사업화 추진 계획이 있을 경우 간결하게 기재합니다. 7) 연구개발 성과의 활용방안 및 기대효과: 연구개발성과의 수요처, 활용내용, 경제적  ...

--- 결과 3 (코사인 유사도: 0.6263) | (전문) 산업기술혁신사업 공통 운영요령(고시 제2024-218호).hwpx ---
[(전문) 산업기술혁신사업 공통 운영요령(고시 제2024-218호) > [별표 5] 산업기술혁신사업 연구개발비 사용용도 및 세부 산정기준] (22/31)
전체 사용목적 등은 사용 불가) · 참여연구자 및 연구근접지원인력이 아닌 자의 연구인력 지원비(단, 연구개발인력 양성을 목적으로 하는 연구개발과제의 경우 지원대상 인력에 대한 비용은 제외한다) 계상불가 · ...

--- 결과 4 (코사인 유사도: 0.6223) | [별지 제5호서식] 최종보고서(국가연구개발혁신법 시행규칙).hwp ---
[[별지 제5호서식] 최종보고서(국가연구개발혁신법 시행규칙)]
16. 연구개발성과: 연구개발과제 수행에 따른 정량적·정성

---

## 5. RAG 체인 구현 (LCEL + Claude)

> Claude API는 `temperature` 파라미터를 받지 않으므로 지정하지 않습니다.

In [14]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

llm = ChatAnthropic(model="claude-sonnet-4-6", max_tokens=2048)

rag_prompt = ChatPromptTemplate.from_template("""
당신은 국가 R&D 사업 규정·지침 전문가입니다. 아래 참고 문서를 바탕으로 질문에 정확하게 답하세요.

규칙:
- 참고 문서에 없는 내용은 "제공된 문서에서 찾을 수 없습니다"라고 답하세요.
- 답변 끝에 근거로 사용한 문서명을 [문서명] 형식으로 표기하세요.
- 조항 번호, 수치, 기한이 있으면 정확히 인용하세요.

[참고 문서]
{context}

[질문]
{question}

[답변]
""")

def format_docs(docs):
    """검색된 Document 목록을 문서명과 함께 하나의 문자열로 합칩니다."""
    return "\n\n".join(f"[문서 {i} - {d.metadata.get('file_name', '알 수 없음')}]\n{d.page_content}"
                        for i, d in enumerate(docs, 1))

rag_chain = (
    RunnableParallel({
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    })
    | rag_prompt
    | llm
    | StrOutputParser()
)
print("RAG 체인 구성 완료")

RAG 체인 구성 완료


In [15]:
# RAG 체인 실행
question = "연구개발비 사용 기준에서 간접비는 어떻게 계상하나요?"
answer = rag_chain.invoke(question)
print(f"질문: {question}\n\n답변:\n{answer}")

질문: 연구개발비 사용 기준에서 간접비는 어떻게 계상하나요?

답변:
## 간접비 계상 기준

### 기본 원칙

**제21조 제2항**에 따라, 연구개발기관의 장은 간접비로 **여러 연구개발과제 수행에 공통적으로 소요되는 비용**으로서 필요한 금액을 계상하여야 합니다.

---

### 계상 금액 산출이 어려운 경우

**제21조 제3항**에 따라, 필요한 금액을 산출하기 어려운 경우에는 **연구개발비 사용에 관한 자체규정**에 따라 해당 금액을 계상하여야 합니다.

> ⚠️ 이 경우, 자체규정에 따라 **국가연구개발사업에 적용되는 금액이 그 외의 경우에 적용되는 금액보다 높아서는 아니 됩니다.**

---

### 계상 시 금지 사항 (제21조 제4항)

간접비 계상 시 아래 비용은 계상할 수 없습니다:

| 금지 유형 | 내용 |
|-----------|------|
| 중복계상 | 동일 비용을 직접비와 간접비로 중복 계상 불가 |
| 내부거래 비용 | 연구개발기관 내부 및 동일 과제 수행 기관 간 발생 비용 (예외 있음) |
| 임차료 | 중앙행정기관의 장이 인정하지 않은 내·외부 연구공간 임차료 |

---

### 간접비 사용 관련 주의사항 (별표 6)

- 협약 시 산정한 금액보다 간접비 총액을 **초과하여 사용 불가** (단, 전문기관의 사전 승인을 득하여 간접비 고시비율 내 증액한 경우 예외)
- **직접비 사용비율이 50% 이하**인 과제의 경우, 간접비 사용비율이 직접비 사용비율을 초과한 금액은 회수 대상
  - (산식) `간접비 총액 × (간접비 사용비율 - 직접비 사용비율)`
- **연구개발능률성과급**은 해당 과제 간접비의 **10%를 초과하여 지급 불가**

---

[국가연구개발사업 연구개발비 사용 기준(과학기술정보통신부고시)(제2025-9호)(20250306).hwp]
[(전문) 산업기술혁신사업 공통 운영요령(고시 제2024-218호).hwpx]


In [16]:
# 다양한 질문으로 테스트 (마지막 질문은 문서에 없을 가능성이 높은 질문)
test_questions = [
    "정부납부기술료는 어떤 경우에 납부하나요?",
    "연구윤리 위반 시 어떤 조치를 받나요?",
    "연구시설·장비비 통합관리기관 지정 절차를 설명해주세요.",
    "오늘 서울 날씨는 어때요?",
]
for q in test_questions:
    print("=" * 60)
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}\n")

Q: 정부납부기술료는 어떤 경우에 납부하나요?
A: ## 정부납부기술료 납부 방법

제공된 문서에서 **정부납부기술료를 납부하는 구체적인 사유(어떤 경우에 납부하는지)**에 대한 내용은 찾을 수 없습니다.

다만, 정부납부기술료의 **납부 방법**에 대해서는 다음과 같이 규정되어 있습니다:

### 납부 방법
- **원칙**: 정부납부기술료는 **현금**으로 납부해야 합니다. (제9조 제1항)

### 분할 납부
- 장관 또는 전문기관의 장은 기술료등납부의무기관의 **신청**에 따라 **최대 3회까지** 분할 납부가 가능합니다. (제9조 제2항)
- 분할 납부 신청 시, 납부기한 내에 분할납부신청서와 함께 다음 중 하나의 **보증 또는 담보**를 제공해야 합니다: (제9조 제3항)
  1. 지급이행보증보험증권
  2. 공증약속어음

### 보증·담보 변경
- 소멸시효 도래 등으로 보증 또는 담보를 변경할 필요가 있는 경우, **전문기관의 승인**을 받아 대체 보증 또는 담보를 제출해야 합니다. (제9조 제4항)

> 정부납부기술료를 **납부해야 하는 구체적인 사유**가 궁금하시다면, 관련 조항(제8조 등)이 포함된 추가 문서를 참고하시기 바랍니다.

[기술료 징수 및 관리에 관한 통합요령(산업통상자원부고시)(제2025-50호)(20250407).hwp]

Q: 연구윤리 위반 시 어떤 조치를 받나요?
A: # 연구윤리 위반 시 조치 내용

## 1. 서울대학교 연구진실성위반행위에 대한 조치

### 판정 기준 (제8조)
연구진실성위반행위의 위반 정도는 다음 사항을 종합적으로 고려하여 판정합니다:
- 위반의 **동기 및 주관적 정황**
- 위반의 **횟수와 범위**
- 위반으로 **얻게 되는 이익**
- **재발방지 노력** 등 사안의 구체적 경위

> ※ 단, 자신의 행위가 위반행위에 해당하지 않는다고 오인한 경우, **정당한 이유가 있는 때에 한하여** 책임을 면할 수 있습니다.

---

## 2. 국가 R&D(산업기술혁신사업) 위반 시 제재 조치

### 

---

## 6. 소스 문서 반환 패턴

답변과 함께 **어떤 문서를 근거로 삼았는지** 반환하면 규정 QA의 신뢰성이 높아집니다.

In [17]:
rag_chain_with_sources = RunnableParallel({
    "answer": rag_chain,        # 기존 RAG 체인의 답변
    "source_docs": retriever,   # 검색된 원본 문서 목록
})

question = "정부납부기술료는 어떤 경우에 납부하나요?"
result = rag_chain_with_sources.invoke(question)

print(f"질문: {question}\n\n=== 답변 ===\n{result['answer']}")
print(f"\n=== 참고 문서 ({len(result['source_docs'])}개) ===")
for i, d in enumerate(result["source_docs"], 1):
    print(f"\n[출처 {i}] {d.metadata['file_name']}")
    print(f"  {d.page_content[:150]}...")

질문: 정부납부기술료는 어떤 경우에 납부하나요?

=== 답변 ===
## 정부납부기술료 납부 방법

제공된 문서에서 정부납부기술료를 **납부하는 경우(사유)**에 대한 직접적인 규정은 찾을 수 없습니다.

다만, 정부납부기술료의 **납부 방법**에 대해서는 다음과 같이 규정되어 있습니다:

### 납부 방법
- **원칙**: 정부납부기술료는 **현금**으로 납부해야 합니다. (제9조 제1항)

### 분할 납부
- 장관 또는 전문기관의 장은 기술료등납부의무기관의 신청에 따라 **최대 3회까지** 분할 납부하도록 할 수 있습니다. (제9조 제2항)
- 분할 납부 시, 납부기한 내에 **분할납부신청서**와 함께 다음 중 하나의 보증 또는 담보를 제공해야 합니다. (제9조 제3항)
  1. 지급이행보증보험증권
  2. 공증약속어음

### 보증·담보 변경
- 소멸시효 도래 등으로 보증·담보를 변경할 필요가 있는 경우, **전문기관의 승인**을 받아 대체 보증 또는 담보를 제출해야 합니다. (제9조 제4항)

> 정부납부기술료를 **납부하여야 하는 사유나 조건**에 대한 내용은 제공된 문서에서 찾을 수 없습니다.

[기술료 징수 및 관리에 관한 통합요령(산업통상자원부고시)(제2025-50호) 제9조]

=== 참고 문서 (5개) ===

[출처 1] [별지 제7호서식] 연구개발비 사용실적보고서(국가연구개발혁신법 시행규칙).hwp
  [[별지 제7호서식] 연구개발비 사용실적보고서(국가연구개발혁신법 시행규칙)]
기재합니다. 2) 발생이자 반납액: 연구개발기간 중 또는 연구개발기간 후 발생한 이자 중 반납하려는 금액(제3호다목5에 따른 국고반납에 해당하는 금액)을 기재합니다. 3) 정부지분율 (1) 현...

[출처 2] 기술료 징수 및 관리에 관한 통합요령(산업통상자원부고시)(제2025-50호)(20250407).hwp
  [기술료 징수 및 관리에 관한 통합요령 > 제2장 기술료 및 정부납부기술료 > 제9조(정부납부기술료 납부방법)]
제9조(정부납부기술료 납부방법) ①

---

## 7. 스트리밍으로 답변 받기

`stream()` 으로 답변이 생성되는 대로 실시간 출력합니다.

In [18]:
question = "연구윤리 위반 시 어떤 조치를 받나요? 자세히 설명해주세요."
print(f"Q: {question}")
print("A: ", end="", flush=True)

for chunk in rag_chain.stream(question):
    print(chunk, end="", flush=True)
print()

Q: 연구윤리 위반 시 어떤 조치를 받나요? 자세히 설명해주세요.
A: # 연구윤리 위반 시 조치 사항

## 1. 산업기술혁신사업 공통 운영요령상 제재 조치

### 📌 참여제한 처분 (주요 위반 유형별 제한 기간)

| 위반 행위 | 제한 기간 |
|-----------|-----------|
| 연구개발비 증명자료 위조·변조 또는 사용내역 거짓 보고 | **2년** |
| 「생명윤리 및 안전에 관한 법률」 제64조~제68조 해당 행위 | **2년** |
| 「연구실 안전환경 조성에 관한 법률」 제43조 해당 행위 | **2년** |
| 조사의 공정성·객관성을 해치는 행위 | **2년** |
| 지식재산권 자료 미제출 또는 다중제출 등 부정 제출 | **1년** |
| 의무 보고서 미제출 또는 허위 작성 | **1년** |
| 시설·장비 임의 처분 (성과활용기간 만료 시 제외) | **2년 이내** |
| 민간투자유치 사업에서 사전 승인 없이 투자계약 변경·무효화·양도 등 | **2년 이내** |

### 📌 병과 가능한 추가 제재 (제44조 제2항)
> "참여제한 처분이나 **제재부가금 부과처분은 병과**할 수 있으며, 이와는 별도로 **이미 지급한 정부 연구개발비 중 제재사유와 관련된 연구개발비를 환수**할 수 있다."

즉, 다음 세 가지 조치가 **동시에** 적용될 수 있습니다:
1. ✅ 참여제한 처분
2. ✅ 제재부가금 부과
3. ✅ 연구개발비 환수

### 📌 제재처분 절차 (제44조 제3항)
- 제재조치 및 연구개발비 환수 관련 사항은 **제재처분평가단**을 구성하여 다음 사항을 검토
  - 제재처분의 필요성
  - 제재처분의 종류 및 수준
  - ※ 단, **보안과제로 분류된 경우** 평가단을 구성하지 않을 수 있음

---

## 2. 서울대학교 연구진실성위원회 규정상 조치

### 📌 연구진실성위반행위 해당 사례 (제5조)
고의 또는 **중대한 과실**로 아래 행위를 한 경우 연구진실성위반행위의 책임을 짐:

1. **부당 연구성과 사용*

In [19]:
# 스트리밍 + 출처 표시: 검색은 한 번만 수행하고, 같은 문서로 답변을 스트리밍한 뒤 출처를 출력
question = "연구시설·장비비 통합관리기관 지정 절차를 설명해주세요."
retrieved = retriever.invoke(question)

stream_chain = rag_prompt | llm | StrOutputParser()
print(f"Q: {question}")
print("A: ", end="", flush=True)
for chunk in stream_chain.stream({"context": format_docs(retrieved), "question": question}):
    print(chunk, end="", flush=True)

print("\n\n[참고 문서]")
for name in dict.fromkeys(d.metadata["file_name"] for d in retrieved):
    print(f"  - {name}")

Q: 연구시설·장비비 통합관리기관 지정 절차를 설명해주세요.
A: 제공된 문서에서 찾을 수 없습니다.

제공된 참고 문서들은 다음 내용을 포함하고 있으나:

- **제102조**: 연구시설·장비비통합관리기관의 **내부 운영규정 마련·운영** 사항
- **제108조**: 연구시설·장비비의 **이관·반납** 절차
- **서울대학교 기준 제11조**: 장비심의위원회의 역할 및 임무
- **표준지침 제14조**: 조사평가 결과관리
- **산업기술혁신사업 운영요령 제13조**: 주관연구개발기관의 권한과 책임

**연구시설·장비비 통합관리기관 지정 절차**(신청 요건, 지정 신청 방법, 심사 기준, 지정 승인 주체 및 절차 등)에 관한 내용은 제공된 문서에 포함되어 있지 않습니다.

해당 내용은 과학기술정보통신부고시 「국가연구개발사업 연구개발비 사용 기준」의 **제7장 관련 조항** 중 지정 절차를 규정한 별도 조문(예: 제100조, 제101조 등)을 확인하시거나, 소관 중앙행정기관 또는 전문기관에 문의하시기 바랍니다.

[문서 1 - 국가연구개발사업 연구개발비 사용 기준(과학기술정보통신부고시)(제2025-9호)]
[문서 2 - 국가연구개발사업 연구개발비 사용 기준(과학기술정보통신부고시)(제2025-9호)]

[참고 문서]
  - 국가연구개발사업 연구개발비 사용 기준(과학기술정보통신부고시)(제2025-9호)(20250306).hwp
  - 붙임2. 「서울대학교 연구시설·장비비 통합관리 기준」 개정 전문 20241101.hwp
  - 국가연구개발 시설·장비의 관리 등에 관한 표준지침 (과학기술정보통신부고시)(제2025-46호)(20250707).hwp
  - (전문) 산업기술혁신사업 공통 운영요령(고시 제2024-218호).hwpx


---

## 핵심 정리

```
HwpHwpxDirectoryLoader → LegalArticleSplitter → HuggingFaceEmbeddings → Milvus
                                                                                  │
질문 → retriever | format_docs ──→ rag_prompt → ChatAnthropic → StrOutputParser → 답변
                       └─ source_docs (출처 반환) / .stream() (실시간 출력)
```

| 컴포넌트 | 역할 |
|----------|------|
| `HwpHwpxDirectoryLoader` | HWP/HWPX 일괄 로드 |
| `LegalArticleSplitter` | 조문(장·조) 인식 청크 분할 |
| `HuggingFaceEmbeddings` | 텍스트 → 384차원 벡터 (로컬) |
| `Milvus` | 벡터 저장 및 유사도 검색 |
| `ChatAnthropic` | Claude 답변 생성 |
| `RunnableParallel` | 답변 + 출처를 함께 반환 |